In [ ]:
import pandas as pd
from pathlib import Path

DATASET_PATH = Path("Datasets")

names = ["Hiruni", "Mineth", "Sineth", "Thanushka", "Thinula"]
activities = ["Bending", "Idle", "Picking", "Pushing"]

activity_labels = {
    "Bending": 1,
    "Idle": 2,
    "Picking": 3,
    "Pushing": 4
}

# Sensor mapping
SENSOR_MAP = {
    "2B_Left": "Left_Wrist",
    "23_Right": "Right_Wrist",
    "2C_Leg": "Upper_Leg"
}

def process_sensor_file(file_path: Path, body_part: str) -> pd.DataFrame:
    """Read CSV, extract FreeAcc columns, rename to Acc, add prefix."""
    df = pd.read_csv(file_path, skiprows=11)
    acc_df = df[["FreeAcc_X", "FreeAcc_Y", "FreeAcc_Z"]].copy()
    acc_df.rename(columns={
        "FreeAcc_X": "Acc_X",
        "FreeAcc_Y": "Acc_Y",
        "FreeAcc_Z": "Acc_Z"
    }, inplace=True)
    acc_df = acc_df.add_prefix(f"{body_part}_")
    return acc_df

# Create output directory
out_dir = Path("merged_persons")
out_dir.mkdir(parents=True, exist_ok=True)

for person in names:
    all_rows = []   
    for activity in activities:
        activity_folder = DATASET_PATH / person / f"{person}_{activity}"
        left_path = activity_folder / "2B_Left.csv"
        right_path = activity_folder / "23_Right.csv"
        leg_path = activity_folder / "2C_Leg.csv"

        # Process each sensor
        left_df  = process_sensor_file(left_path, SENSOR_MAP["2B_Left"])
        right_df = process_sensor_file(right_path, SENSOR_MAP["23_Right"])
        leg_df   = process_sensor_file(leg_path, SENSOR_MAP["2C_Leg"])

        # Concatenate horizontally
        merged = pd.concat([left_df, right_df, leg_df], axis=1)

        # Add Activity column 
        merged["Activity"] = activity_labels[activity]

        all_rows.append(merged)

    # Concatenate all activities vertically 
    person_df = pd.concat(all_rows, ignore_index=True)

    # Save one CSV per person
    out_file = out_dir / f"{person}.csv"
    person_df.to_csv(out_file, index=False)
    print(f"Saved: {out_file}")

Saved: merged_persons\Hiruni.csv
Saved: merged_persons\Mineth.csv
Saved: merged_persons\Sineth.csv
Saved: merged_persons\Thanushka.csv
Saved: merged_persons\Thinula.csv
